# Research: Anthropic at $83, section by section

This notebook follows the blog post (`outline.md`) in order. For each section it re-checks every number the post quotes against `data/` (or against `montecarlo.py`, for numbers that 09 and 10 compute from the model), then shows that section's charts. **Re-run it after re-running any analysis notebook.** A failed `check` means the post and the analysis disagree: fix the post, or find out why the analysis moved.

**The post's two anchors** (outline, top):
- **Own contracts, ~$18–45 a share** (09 §6): what the revenue Anthropic's signed compute can serve is worth. The only forecast the company has put money behind.
- **Our forecast, ~$69 average** (08): 20,000 simulated worlds with Anthropic taking 30–55% of AI spending. Higher than the contracts. That's a judgment, and the post says whose (§3).

`check(label, value, expected, tol)` prints the value and asserts it's within `tol` of what the post says. Money: $ billions unless the label says otherwise.

In [1]:
import numpy as np
import pandas as pd
from common import *
import montecarlo as mc

B = 1000
def check(label, value, expected, tol):
    ok = abs(value - expected) <= tol
    print(f"{'ok ' if ok else 'BAD'}  {label}: {value:,.3f}  (post: {expected:,.3f} ± {tol})")
    assert ok, f'{label}: post says {expected}, analysis gives {value:.3f}'

X = load('valuation_inputs').set_index('item').value
VAL = load('valuation').query("metric == 'per_share'").set_index('percentile').value
R = wide(load('real_earnings'), index='metric')
IS = wide(load('income_statement'))
K = load('kpis').drop_duplicates('metric').set_index('metric').value
price, fd = X['ipo_price'], mc.FD_SHARES
base = mc.run()                                   # 08's 20,000 worlds (09 §1 asserts the module matches 08)
w = base['worlds']
contracts = {k: one['worlds'].iloc[0] for k, one in mc.contract_worlds().items()}
c_val = sorted(c.per_share for c in contracts.values())          # [Q3 efficiency, Q2 efficiency]
prize_ps = mc.PARAMS['prize_value'] * 1e6 / fd                    # $10T per share
implied = lambda v: (price - v) / prize_ps                        # 10 §3: unexplained ÷ $10T

## 0. Hook + TL;DR
- Anthropic priced at **$83** (range $70–78): **~$1.36T** on today's ~16.4B fully diluted shares.
- On its own compute contracts, worth **~$18–45 a share**. On our forecast, averaged over 20,000 worlds, **~$69**; **~33%** of worlds are worth $83.
- $83 implies a **~2%** chance of a $10T breakthrough on our forecast, **~6–11%** on the contracts; SpaceX **~16%** (10 §3, checked in §8).
- **Day one:** ANT opened at **$111.50** (34% above $83). At the open the implied chance is ~7% (forecast), 11–15% (contracts).

In [2]:
check('IPO price, $', price, 83, 0)
check('market value on today\'s shares, $T', price * fd / 1e12, 1.36, 0.01)
check('own contracts, low ($/share)', c_val[0], 18, 0.6)
check('own contracts, high ($/share)', c_val[1], 45, 0.6)
check('our forecast, average ($/share)', VAL['mean'], 69, 0.6)
check('our forecast, average (module = 08)', w.per_share.mean(), VAL['mean'], 0.05)
check('share of worlds worth >= $83', (w.per_share >= price).mean(), 0.33, 0.006)
check('implied breakthrough chance, our forecast', implied(VAL['mean']), 0.022, 0.001)
check('implied chance, contracts at Q2 efficiency', implied(c_val[1]), 0.061, 0.001)
check('implied chance, contracts at Q3 efficiency', implied(c_val[0]), 0.107, 0.001)
first = X['ant_first_trade']                                       # 10 §3: day-one open (press)
implied_at = lambda v, p: (p - v) / prize_ps
check('day-one open, $', first, 111.5, 0)
check('implied chance at the open, forecast', implied_at(VAL['mean'], first), 0.069, 0.001)
check('implied chance at the open, contracts Q2', implied_at(c_val[1], first), 0.108, 0.001)
check('implied chance at the open, contracts Q3', implied_at(c_val[0], first), 0.154, 0.001)

ok   IPO price, $: 83.000  (post: 83.000 ± 0)
ok   market value on today's shares, $T: 1.359  (post: 1.360 ± 0.01)
ok   own contracts, low ($/share): 17.551  (post: 18.000 ± 0.6)
ok   own contracts, high ($/share): 45.480  (post: 45.000 ± 0.6)
ok   our forecast, average ($/share): 69.474  (post: 69.000 ± 0.6)
ok   our forecast, average (module = 08): 69.474  (post: 69.474 ± 0.05)
ok   share of worlds worth >= $83: 0.326  (post: 0.330 ± 0.006)
ok   implied breakthrough chance, our forecast: 0.022  (post: 0.022 ± 0.001)
ok   implied chance, contracts at Q2 efficiency: 0.061  (post: 0.061 ± 0.001)
ok   implied chance, contracts at Q3 efficiency: 0.107  (post: 0.107 ± 0.001)
ok   day-one open, $: 111.500  (post: 111.500 ± 0)
ok   implied chance at the open, forecast: 0.069  (post: 0.069 ± 0.001)
ok   implied chance at the open, contracts Q2: 0.108  (post: 0.108 ± 0.001)
ok   implied chance at the open, contracts Q3: 0.154  (post: 0.154 ± 0.001)


## 1. Start with the S-1, not the headlines (02, 04)
- The **$104B** 9M-2026 net loss is more than entirely valuation marks (**$94.3B** Amazon notes + **$11.0B** share-purchase commitments).
- Clean revenue: $33.2B reported, ~6% policy-violator revenue out → Q3 run-rate **~$64B**, not $68B.
- Real operating margin **−4%** (9M-2026) vs the company's **15%** "Adjusted" (Q3).
- FCF **$3.6B**, of which **$2.4B** is interest; owner FCF **−$4.1B**.
- **~16.4B** fully diluted shares, not the 2.1B EPS denominator.

In [3]:
is9 = IS['9M-2026']
notes_fv = -is9['Change in fair value of convertible notes, derivative liabilities, and SAFEs']
check('net loss 9M-2026', -is9['Net loss'] / B, 104.0, 0.1)
check('fair-value change on notes and other instruments', notes_fv / B, 94.3, 0.05)
# The $11.0B partner share-commitment mark and the 101% share of the loss are built up in 02 §1.
check('reported revenue 9M-2026', R.loc['revenue', '9M-2026'] / B, 33.2, 0.05)
check('Q3 clean run-rate', R.loc['clean_revenue', 'Q3-2026'] * 4 / B, 64, 0.5)
check('Q3 reported run-rate', R.loc['revenue', 'Q3-2026'] * 4 / B, 68, 0.5)
check('real operating margin 9M-2026', R.loc['real_operating_income', '9M-2026'] / R.loc['revenue', '9M-2026'], -0.04, 0.005)
check('company "Adjusted" operating margin Q3', R.loc['adjusted_operating_income', 'Q3-2026'] / R.loc['revenue', 'Q3-2026'], 0.15, 0.006)
check('FCF 9M-2026', R.loc['fcf', '9M-2026'] / B, 3.6, 0.05)
check('interest income 9M-2026', R.loc['interest_income', '9M-2026'] / B, 2.4, 0.05)
check('owner FCF 9M-2026', R.loc['owner_fcf', '9M-2026'] / B, -4.1, 0.05)
check('fully diluted shares, B', fd / 1e9, 16.4, 0.05)
check('EPS denominator, B', is9['Weighted-average shares outstanding, basic and diluted'] / 1e6, 2.1, 0.05)

ok   net loss 9M-2026: 104.039  (post: 104.000 ± 0.1)
ok   fair-value change on notes and other instruments: 94.324  (post: 94.300 ± 0.05)
ok   reported revenue 9M-2026: 33.242  (post: 33.200 ± 0.05)
ok   Q3 clean run-rate: 63.755  (post: 64.000 ± 0.5)
ok   Q3 reported run-rate: 67.824  (post: 68.000 ± 0.5)
ok   real operating margin 9M-2026: -0.038  (post: -0.040 ± 0.005)
ok   company "Adjusted" operating margin Q3: 0.145  (post: 0.150 ± 0.006)
ok   FCF 9M-2026: 3.601  (post: 3.600 ± 0.05)
ok   interest income 9M-2026: 2.359  (post: 2.400 ± 0.05)
ok   owner FCF 9M-2026: -4.099  (post: -4.100 ± 0.05)
ok   fully diluted shares, B: 16.378  (post: 16.400 ± 0.05)
ok   EPS denominator, B: 2.077  (post: 2.100 ± 0.05)


![02_gaap_loss_waterfall.png](charts/02_gaap_loss_waterfall.png)

![02_revenue_quarterly.png](charts/02_revenue_quarterly.png)

![02_operating_margin.png](charts/02_operating_margin.png)

![02_fcf_bridge.png](charts/02_fcf_bridge.png)

![04_share_count.png](charts/04_share_count.png)

## 1b. Is it a good business per token? (05)
- Serving costs **~14.5¢** per revenue dollar (**~85%** gross margin); R&D is **58¢**.
- Gross profit **$28.4B** already covers R&D **$19.2B**.
- Subscription vs API margins, cost per token: from 05 (`data/token_economics.csv`); checked in 05.

In [4]:
F = wide(load('opex_by_function'))['9M-2026']
rev9 = R.loc['revenue', '9M-2026']
check('cost of revenue per $ of revenue', F['Cost of revenue'] / rev9, 0.145, 0.0015)
check('R&D per $ of revenue', F['Research and development'] / rev9, 0.58, 0.005)
check('gross profit 9M-2026', (rev9 - F['Cost of revenue']) / B, 28.4, 0.05)
check('R&D 9M-2026', F['Research and development'] / B, 19.2, 0.05)

ok   cost of revenue per $ of revenue: 0.145  (post: 0.145 ± 0.0015)
ok   R&D per $ of revenue: 0.579  (post: 0.580 ± 0.005)
ok   gross profit 9M-2026: 28.430  (post: 28.400 ± 0.05)
ok   R&D 9M-2026: 19.240  (post: 19.200 ± 0.05)


![05_cost_per_revenue_dollar.png](charts/05_cost_per_revenue_dollar.png)

![05_margin_vs_usage.png](charts/05_margin_vs_usage.png)

## 2. The $895B of compute commitments, and the contracts view (06, 09 §6)
- **$895B** of commitments; **~$448B** due before 2031.
- The contracts imply a ceiling of **~$236B** a year (Q3 efficiency; ~$304B at Q2's), below **~97%** of our worlds.
- They can serve **~$262–325B** of 2030 revenue, worth **~$18–45** a share.

In [5]:
check('commitments, all in', K['commitments_all_in'] / B, 895, 0.5)
check('due before 2031', K['commitments_before_2031'] / B, 448, 0.5)
ceil = sorted(c.ceiling for c in contracts.values())
check('contracts ceiling at Q3 efficiency', ceil[0] / B, 236, 1)
check('contracts ceiling at Q2 efficiency', ceil[1] / B, 304, 1)
check('our worlds with a ceiling above the Q3 contracts ceiling', (w.ceiling > ceil[0]).mean(), 0.97, 0.005)
rev30 = sorted(c.revenue_2030 for c in contracts.values())
check('contracts revenue 2030, low', rev30[0] / B, 262, 1)
check('contracts revenue 2030, high', rev30[1] / B, 325, 1)

ok   commitments, all in: 895.400  (post: 895.000 ± 0.5)
ok   due before 2031: 447.500  (post: 448.000 ± 0.5)
ok   contracts ceiling at Q3 efficiency: 236.172  (post: 236.000 ± 1)
ok   contracts ceiling at Q2 efficiency: 303.854  (post: 304.000 ± 1)
ok   our worlds with a ceiling above the Q3 contracts ceiling: 0.972  (post: 0.970 ± 0.005)
ok   contracts revenue 2030, low: 262.151  (post: 262.000 ± 1)
ok   contracts revenue 2030, high: 324.750  (post: 325.000 ± 1)


![06_commitments_by_partner.png](charts/06_commitments_by_partner.png)

![09_contracts_view.png](charts/09_contracts_view.png)

### Sidebar: where Claude runs, and who gets paid (07)
Megawatts and $ per MW-year are outside data (`data/compute_capacity.csv`, with `reliability` flags), checked in 07. Treat totals as ±25%.

![07_compute_mw_by_chip.png](charts/07_compute_mw_by_chip.png)

![07_compute_mw_by_year.png](charts/07_compute_mw_by_year.png)

![07_price_per_mw.png](charts/07_price_per_mw.png)

## 3. Forecasting from their own claims (06)
- Ceiling = the S-1's **$40T** × AI's share of wages (1.5–8%) × Anthropic's share of AI (**30–55%**).
- The company's ~$81B October run-rate sits at our **P18**: the forecast starts on the high side.

**Where I part ways with the contracts (author's view; edit to taste).** The contracts are the plan Anthropic has paid for so far, not the most it can sell. It keeps signing (~$100B more after Sep 30 alone), and it is short of compute in most of our worlds (07 §4). I think Anthropic can hold 30–55% of AI spending, and that's why our forecast sits above its own contracts. That's my judgment, not something the S-1 shows: at 15–35%, the same model gives an average of ~$44. Readers who don't share it should anchor on the contracts.

In [6]:
check('Anthropic share of AI, low', mc.PARAMS['anthropic_share_of_ai'][0], 0.30, 0)
check('Anthropic share of AI, high', mc.PARAMS['anthropic_share_of_ai'][1], 0.55, 0)
check('knowledge-work wages, $T', K['knowledge_work_value'] / 1e6, 40, 0.5)
q4 = base['revenue']['Q4-2026'] * 4
check('October run-rate percentile in our worlds', (q4 < X['run_rate_revenue_oct2026']).mean(), 0.18, 0.01)
check('average with the old 15-35% share', mc.run(anthropic_share_of_ai=(0.15, 0.35))['worlds'].per_share.mean(), 44, 1)

ok   Anthropic share of AI, low: 0.300  (post: 0.300 ± 0)
ok   Anthropic share of AI, high: 0.550  (post: 0.550 ± 0)
ok   knowledge-work wages, $T: 40.000  (post: 40.000 ± 0.5)
ok   October run-rate percentile in our worlds: 0.178  (post: 0.180 ± 0.01)


ok   average with the old 15-35% share: 43.916  (post: 44.000 ± 1)


![06_gompertz_explainer.png](charts/06_gompertz_explainer.png)

![06_growth_vs_size.png](charts/06_growth_vs_size.png)

## 4. The Monte Carlo: average and median (06, 08)
- Median revenue **~$155B** (2027), **~$524B** (2030); P10–P90 2030 **$306–907B**.
- Median real operating margin **+5% / +13% / +18%** (2027/28/30).
- **Average ~$69; median ~$59; P10–P90 $10–143.** ~4% of worlds are worth $0.
- Price ladder: Series G $25.91 → contracts $18–45 → Series H $58.90 ≈ median → average $69 → Sep-30 notes mark ~$71 → range $70–78 → $83.
- Buying at $83 in the median world earns **~8.3%** a year vs 10.5% required (08 §6).

In [7]:
rev = base['revenue']
check('median revenue 2027', rev['2027'].median() / B, 155, 1)
check('median revenue 2030', rev['2030'].median() / B, 524, 2)
check('P10 revenue 2030', rev['2030'].quantile(0.1) / B, 306, 2)
check('P90 revenue 2030', rev['2030'].quantile(0.9) / B, 907, 3)
m = (base['real_operating_income'] / rev).median()
for y, e in (('2027', 0.05), ('2028', 0.13), ('2030', 0.18)):
    check(f'median real operating margin {y}', m[y], e, 0.006)
check('average value per share', VAL['mean'], 69, 0.6)
check('median value per share', VAL['P50'], 59, 0.6)
check('P10 value per share', VAL['P10'], 10, 0.6)
check('P90 value per share', VAL['P90'], 143, 0.6)
check('worlds worth $0', (w.per_share == 0).mean(), 0.04, 0.006)
pref = load('preferred_stock').query("as_of == '2026-09-30'").set_index('series').issue_price_per_common_usd
notes_mark = wide(load('balance_sheet'))['2026-09-30']['Convertible notes payable'] * 1e6 / load('convertible_notes_outstanding')['Class N Shares Issuable on Conversion'].sum()
check('Series G, $', pref['Series G-1'], 25.91, 0.005)
check('Series H, $', pref['Series H-1'], 58.90, 0.005)
check('Sep-30 notes mark, $', notes_mark, 71, 0.5)
check('discount rate', base['discount_rate'], 0.105, 0.001)

ok   median revenue 2027: 155.485  (post: 155.000 ± 1)
ok   median revenue 2030: 524.111  (post: 524.000 ± 2)
ok   P10 revenue 2030: 305.624  (post: 306.000 ± 2)
ok   P90 revenue 2030: 907.312  (post: 907.000 ± 3)
ok   median real operating margin 2027: 0.048  (post: 0.050 ± 0.006)
ok   median real operating margin 2028: 0.128  (post: 0.130 ± 0.006)
ok   median real operating margin 2030: 0.177  (post: 0.180 ± 0.006)
ok   average value per share: 69.474  (post: 69.000 ± 0.6)
ok   median value per share: 59.097  (post: 59.000 ± 0.6)
ok   P10 value per share: 10.301  (post: 10.000 ± 0.6)
ok   P90 value per share: 143.022  (post: 143.000 ± 0.6)
ok   worlds worth $0: 0.038  (post: 0.040 ± 0.006)
ok   Series G, $: 25.914  (post: 25.910 ± 0.005)
ok   Series H, $: 58.901  (post: 58.900 ± 0.005)
ok   Sep-30 notes mark, $: 70.908  (post: 71.000 ± 0.5)
ok   discount rate: 0.105  (post: 0.105 ± 0.001)


![06_revenue_fan.png](charts/06_revenue_fan.png)

![06_margin_fcf.png](charts/06_margin_fcf.png)

![08_value_per_share.png](charts/08_value_per_share.png)

![08_value_sources.png](charts/08_value_sources.png)

## 4b. Your slice: stock comp, donations and dilution (02, 04, 09 §7)
- Stock comp + donations were **~16%** of revenue in 9M-2026 ($3.3B + $2.1B).
- An **$8.2B** stock-comp charge when the IPO lands; a **$32.3B** RSU backlog.
- New grants (4% of revenue) and donations (3%), charged as costs, take **~$36 a share**: a third of what it would be worth without them (~$105).
- The Public Benefit Fund's top-ups (up to 2% more shares per 50% rise from $83) cost **~1%** on average; P90 $143 → $140.
- Footnote: the prospectus also counts a 553M-share Vested Matching Obligation (~16.9B pre-IPO). We charge it as donations instead; counted as shares, the average is ~$67.

In [8]:
check('stock comp + donations, share of revenue 9M-2026', (R.loc['sbc', '9M-2026'] + R.loc['donations', '9M-2026']) / rev9, 0.16, 0.005)
check('stock-comp catch-up at IPO', K['rsu_sbc_catchup_at_ipo'] / B, 8.2, 0.05)
check('RSU backlog', K['rsu_unrecognized_sbc'] / B, 32.3, 0.05)
no_gifts = mc.run(sbc_new_grants_pct=0, donations_pct=0)['worlds'].per_share
check('average without new grants or donations, $', no_gifts.mean(), 105, 1)
check('what grants + donations cost, $/share', no_gifts.mean() - w.per_share.mean(), 36, 1)
pb = w.per_share / mc.public_benefit_multiple(w.per_share.values)
check('Public Benefit top-ups, change in average', pb.mean() / w.per_share.mean() - 1, -0.01, 0.002)
check('P90 after top-ups, $', pb.quantile(0.9), 140, 0.6)
vmo = K['vested_matching_obligation_shares'] * 1e6                # 04 §2: prospectus counts it; we charge it as donations
check('prospectus pre-IPO count, B', (fd + vmo) / 1e9, 16.9, 0.05)
check('average if the 553M are counted as shares, $', VAL['mean'] * fd / (fd + vmo), 67, 0.5)

ok   stock comp + donations, share of revenue 9M-2026: 0.161  (post: 0.160 ± 0.005)
ok   stock-comp catch-up at IPO: 8.200  (post: 8.200 ± 0.05)
ok   RSU backlog: 32.300  (post: 32.300 ± 0.05)


ok   average without new grants or donations, $: 105.038  (post: 105.000 ± 1)
ok   what grants + donations cost, $/share: 35.564  (post: 36.000 ± 1)
ok   Public Benefit top-ups, change in average: -0.009  (post: -0.010 ± 0.002)
ok   P90 after top-ups, $: 140.223  (post: 140.000 ± 0.6)
ok   prospectus pre-IPO count, B: 16.931  (post: 16.900 ± 0.05)
ok   average if the 553M are counted as shares, $: 67.204  (post: 67.000 ± 0.5)


![04_future_dilution.png](charts/04_future_dilution.png)

## 5. What $83 needs without a miracle (09)
One dial at a time from a ~$69 central world: a ~$730B ceiling, or 2.12x revenue per compute dollar (Q2 was 2.08x, Q3 1.72x), or opex ~21% of revenue by 2040, or a ~9.3% discount rate. Checked in 09 §4 (it solves for each). On the contracts view, $83 is ~2–5x the $18–45.

In [9]:
check('$83 ÷ contracts, high end', price / c_val[1], 1.8, 0.05)
check('$83 ÷ contracts, low end', price / c_val[0], 4.7, 0.05)

ok   $83 ÷ contracts, high end: 1.825  (post: 1.800 ± 0.05)
ok   $83 ÷ contracts, low end: 4.729  (post: 4.700 ± 0.05)


![09_ipo_price_map.png](charts/09_ipo_price_map.png)

## 6. The gap is a lottery ticket: the $10T payday (10 §2–3)
- Our forecast covers **~84%** of the price: $1.14T of business, **$0.22T** unexplained.
- The contracts view covers **21–55%**: **$0.61–1.07T** unexplained.
- Implied chance of a $10T prize: **~2%** (forecast), **~6–11%** (contracts).

In [10]:
mv = price * fd / 1e6                                                 # $M
check('forecast: business value, $T', VAL['mean'] * fd / 1e12, 1.14, 0.01)
check('forecast: unexplained, $T', (price - VAL['mean']) * fd / 1e12, 0.22, 0.01)
check('forecast: share of price covered', VAL['mean'] / price, 0.84, 0.005)
check('contracts: share covered, low', c_val[0] / price, 0.21, 0.005)
check('contracts: share covered, high', c_val[1] / price, 0.55, 0.005)
check('contracts: unexplained, low ($T)', (price - c_val[1]) * fd / 1e12, 0.61, 0.01)
check('contracts: unexplained, high ($T)', (price - c_val[0]) * fd / 1e12, 1.07, 0.01)

ok   forecast: business value, $T: 1.138  (post: 1.140 ± 0.01)
ok   forecast: unexplained, $T: 0.222  (post: 0.220 ± 0.01)
ok   forecast: share of price covered: 0.837  (post: 0.840 ± 0.005)
ok   contracts: share covered, low: 0.211  (post: 0.210 ± 0.005)
ok   contracts: share covered, high: 0.548  (post: 0.550 ± 0.005)
ok   contracts: unexplained, low ($T): 0.614  (post: 0.610 ± 0.01)
ok   contracts: unexplained, high ($T): 1.072  (post: 1.070 ± 0.01)


![10_price_vs_dcf.png](charts/10_price_vs_dcf.png)

## 7. How much to bet: Kelly (10 §4)
- Forecast: no position below **~2.2%**; 5% → **~11%** of a portfolio (half-Kelly ~6%); 10% → ~26%.
- If the $10T lands ~2035 (~$4T today), break-even **~5.5%**.
- Contracts: break-even **~6%** (Q2 efficiency) to **~11%** (Q3).
- As a maximum price: no prize ~$69; 5% → ~$100; 10% → ~$131.

Kelly fractions are computed and tabled in 10 §4; the break-evens and maximum prices are simple enough to re-check here.

In [11]:
check('break-even, forecast', implied(VAL['mean']), 0.022, 0.001)
pv4 = 4e6 * 1e6 / fd                     # 10 §4 rounds $10T-in-2035 (~$4.4T today at 10.45%) to $4T
check('break-even, forecast, $10T in 2035', (price - VAL['mean']) / pv4, 0.055, 0.003)
check('break-even, contracts at Q2 efficiency', implied(c_val[1]), 0.061, 0.001)
check('break-even, contracts at Q3 efficiency', implied(c_val[0]), 0.107, 0.001)
check('max price at 5%', VAL['mean'] + 0.05 * prize_ps, 100, 0.6)
check('max price at 10%', VAL['mean'] + 0.10 * prize_ps, 131, 0.6)

ok   break-even, forecast: 0.022  (post: 0.022 ± 0.001)
ok   break-even, forecast, $10T in 2035: 0.055  (post: 0.055 ± 0.003)
ok   break-even, contracts at Q2 efficiency: 0.061  (post: 0.061 ± 0.001)
ok   break-even, contracts at Q3 efficiency: 0.107  (post: 0.107 ± 0.001)
ok   max price at 5%: 100.004  (post: 100.000 ± 0.6)
ok   max price at 10%: 130.534  (post: 131.000 ± 0.6)


![10_kelly.png](charts/10_kelly.png)

## 8. Versus SpaceX (10 §1–3)
- EV / trailing revenue: Anthropic **24x** (35x fully diluted) vs SpaceX **59x**. Growth 12.3x vs 2.6x.
- Our DCF covers ~84% of Anthropic's price (21–55% on contracts) vs **~16%** of SpaceX's ($23.64 vs $143.50, 6.1x).
- Implied breakthrough odds: SpaceX **~16%**, 1.5–7x Anthropic's.

The multiples and SpaceX's re-valued DCF come from 10 (which asserts its 10-Q inputs against EDGAR's XBRL). Here we re-check what can be done from `data/comps_spacex.csv` alone, plus the $23.64.
- SpaceX's AI segment: one unnamed **Customer B** is **two-thirds** of it (~$5.0B of $7.56B in Q3; our inference: Anthropic). Everything else, Grok, Cursor, X, is **~$10B a year**.

In [12]:
S = load('comps_spacex').set_index('item').value
check('SpaceX revenue growth Q3 y/y, x', S['revenue_q3_2026'] / S['revenue_q3_2025'], 2.6, 0.05)
check('Anthropic revenue growth Q3 y/y, x', R.loc['revenue', 'Q3-2026'] / R.loc['revenue', 'Q3-2025'], 12.3, 0.05)
check('SpaceX market cap, $T', S['price'] * S['shares_outstanding'] / 1e6, 1.95, 0.005)
sx_dcf = 23.64                                                  # 10 §2 (re-values the SpaceX post's dcf.py)
sx_chance = (S['price'] - sx_dcf) * S['shares_outstanding'] / (mc.PARAMS['prize_value'])
check('SpaceX: share of price our DCF covers', sx_dcf / S['price'], 0.16, 0.005)
check('SpaceX implied chance', sx_chance, 0.163, 0.002)
check('SpaceX ÷ Anthropic (forecast)', sx_chance / implied(VAL['mean']), 7.3, 0.1)
check('SpaceX ÷ Anthropic (contracts, Q3)', sx_chance / implied(c_val[0]), 1.5, 0.05)
cust_b = S['customer_b_pct_q3_2026'] / 100 * S['revenue_q3_2026']
check('Customer B, share of SpaceX AI segment Q3', cust_b / S['revenue_si_q3_2026'], 0.67, 0.005)
check('SpaceX AI segment ex Customer B, x4, $B', (S['revenue_si_q3_2026'] - cust_b) * 4 / B, 10, 0.2)

ok   SpaceX revenue growth Q3 y/y, x: 2.614  (post: 2.600 ± 0.05)
ok   Anthropic revenue growth Q3 y/y, x: 12.269  (post: 12.300 ± 0.05)
ok   SpaceX market cap, $T: 1.949  (post: 1.950 ± 0.005)
ok   SpaceX: share of price our DCF covers: 0.165  (post: 0.160 ± 0.005)
ok   SpaceX implied chance: 0.163  (post: 0.163 ± 0.002)
ok   SpaceX ÷ Anthropic (forecast): 7.349  (post: 7.300 ± 0.1)
ok   SpaceX ÷ Anthropic (contracts, Q3): 1.519  (post: 1.500 ± 0.05)
ok   Customer B, share of SpaceX AI segment Q3: 0.665  (post: 0.670 ± 0.005)
ok   SpaceX AI segment ex Customer B, x4, $B: 10.127  (post: 10.000 ± 0.2)


![10_multiples_vs_spacex.png](charts/10_multiples_vs_spacex.png)

![07_capacity_vs_need.png](charts/07_capacity_vs_need.png)

## 9. Verdict
$83 is ~2–5x what Anthropic's own contracts support ($18–45) and 1.2x our forecast ($69). The rest is a top-third business outcome on our forecast, or a breakthrough ticket at ~2% (forecast) to ~6–11% (contracts). Not a buy call. Next to SpaceX, it is the more fairly priced trillion-dollar IPO, and its unexplained part rests on the breakthrough it is better placed to win.

**Not yet checked here:** 05's subscription/API margins and cost per token, 07's megawatts and $ per MW-year, 09 §4's one-dial answers, 10's multiples and Kelly fractions. Each is computed (and where possible asserted) in its own notebook. Add checks here if the post quotes them.